# Data Exploration

## Configurações Iniciais

### Configurações a depender do ambiente

In [1]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/1_bronze_data/"
    SAVE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/2_silver_data/"
else:
    BASE_PATH = "../1_bronze_data/"
    SAVE_PATH = "../2_silver_data/"

Detectado: 💻 Ambiente Local (WSL/Jupyter)
Versão do Python: 3.13.3


### Importação de Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "browser"

### Importação de Dados

In [3]:
df = pd.read_csv(
    f"{BASE_PATH}/b_incidentes.csv",
    sep=";",
    encoding="ISO-8859-1",
    low_memory=False
)

FileNotFoundError: [Errno 2] No such file or directory: '../1_bronze_data//b_incidentes.csv'

In [ ]:
df.head(20)

In [ ]:
# Equivalente ao printSchema() do PySpark
df.info()

In [ ]:
# Converter colunas temporais para datetime
for col in ["Aberto", "Resolvido", "Encerrado"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [ ]:
# Tamanho do DataFrame
total_registros = len(df)
colunas_originais = df.columns.tolist()
numero_colunas = len(colunas_originais)
print(f"Total de Registros: {total_registros}")
print(f"Total de Colunas: {numero_colunas}")

## Exploração de Dados

### Sobre o DF

In [ ]:
coluna_chave = "Número"

colunas_categoricas = [
    "Prioridade", "Produto", "Categoria", "Subcategoria",
    "Grupo designado", "Item de configuração", "Descrição Resumida",
    "Solução", "Aberto por", "Status", "Incidente Pai"
]

colunas_numericas = [
    "Duração",
    "Tempo_Pos_Resolução",
    "Diferença_Limite_Duração"
]

colunas_temporais = ["Aberto", "Resolvido", "Encerrado"]

colunas_validacao = [
    "Duração_is_inconsistent", "Modified_Record",
    "Resolvido_after_Encerrado", "Entrou para KPI?", "KPI Violado?"
]

total_colCat  = len(colunas_categoricas)
total_colNum  = len(colunas_numericas)
total_colTemp = len(colunas_temporais)
total_colVal  = len(colunas_validacao)
total_col = total_colCat + total_colNum + total_colTemp + total_colVal + 1

if total_col == numero_colunas:
    print(f"Total de Colunas: {total_col}")
    print(f"  Numéricas:   {total_colNum}")
    print(f"  Temporais:   {total_colTemp}")
    print(f"  Validação:   {total_colVal}")
    print(f"  Categóricas: {total_colCat}")
else:
    print(f"Faltam classificar {numero_colunas - total_col} colunas, revise.")

### Período Temporal

In [ ]:
data_minima_Aberto    = df["Aberto"].min()
data_maxima_Aberto    = df["Aberto"].max()
data_minima_Resolvido = df["Resolvido"].min()
data_maxima_Resolvido = df["Resolvido"].max()
data_minima_Encerrado = df["Encerrado"].min()
data_maxima_Encerrado = df["Encerrado"].max()

data_minima = min(data_minima_Aberto, data_minima_Resolvido, data_minima_Encerrado)
data_maxima = max(data_maxima_Aberto, data_maxima_Resolvido, data_maxima_Encerrado)
intervalo_total = (data_maxima - data_minima).days

print(f"Data Mínima: {data_minima}")
print(f"Data Máxima: {data_maxima}")
print(f"Intervalo Total: {intervalo_total} dias")

### Dados categóricos

In [ ]:
freq_dict = {}

for col in colunas_categoricas:
    freq_dict[col] = (
        df.groupby(col, dropna=False)
        .size()
        .reset_index(name="count")
        .assign(percentual=lambda x: (x["count"] / total_registros * 100).round(2))
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

print(f"Frequência por valor gerada para as colunas: {list(freq_dict.keys())}")

In [ ]:
stats_data = []

for col_name in colunas_categoricas:
    nulls    = df[col_name].isna().sum()
    distintos = df[col_name].nunique()
    pct_nulls = round(nulls / total_registros * 100, 2) if total_registros > 0 else 0.0
    stats_data.append({
        "coluna": col_name,
        "total_registros": total_registros,
        "distintos": distintos,
        "nulos": int(nulls),
        "pct_nulls": pct_nulls,
    })

df_info_cat = pd.DataFrame(stats_data).sort_values(["nulos","distintos"]).reset_index(drop=True)
display(df_info_cat)

In [ ]:
display(freq_dict['Aberto por'])

In [ ]:
display(freq_dict['Prioridade'])

In [ ]:
display(freq_dict['Status'])

In [ ]:
display(freq_dict['Grupo designado'])

*Informações gerais*

- A maior parte dos incidentes são P4, seguidos por P3 e P2. As classes minoritárias são P5 e P1 - com menos de 1% de representatividade na base. Inclusive, no período analisado, só teve 1 P1.

- Não há muitos incidentes abertos de forma manual, isso é um bom indicador. Precisamos entender por que e o que faz um incidente ser aberto manualmente.

- A maior parte dos incidentes são encerrados automaticamente ou não têm intervenção.

- A esmagadora maioria dos incidentes é resolvida pelo Time 14.

#### Prioridade

In [ ]:
# Mapeamento ordinal da prioridade
prioridade_map = {
    "1 - Crítica":    1,
    "2 - Alta":       2,
    "3 - Média":      3,
    "4 - Baixa":      4,
    "5 - Muito Baixa":5,
}
df["Prioridade_Ordinal"] = df["Prioridade"].map(prioridade_map)

In [ ]:
# O incidente único P1
p1 = df[df["Prioridade"] == "1 - Crítica"]
display(p1)

#### Incidente Pai

In [ ]:
# Contagem de filhos por incidente pai
df_contagem_filhos = (
    df[df["Incidente Pai"].notna()]
    .groupby("Incidente Pai")
    .size()
    .reset_index(name="Total_Filhos")
    .rename(columns={"Incidente Pai": "ID_Pai_Referencia"})
)

df = df.merge(df_contagem_filhos, left_on="Número", right_on="ID_Pai_Referencia", how="left")
df["Total_Filhos"] = df["Total_Filhos"].fillna(0).astype(int)
df["Eh_Pai"] = df["Total_Filhos"] > 0
df.drop(columns=["ID_Pai_Referencia"], inplace=True)

In [ ]:
# Validando pais com mais de 1 filho
display(df[df["Eh_Pai"]]["Total_Filhos"].describe())

In [ ]:
# Validando incidentes pais que também são filhos
print((df["Eh_Pai"] & df["Incidente Pai"].notna()).sum())

#### Grupo designado

In [ ]:
# Moda de prioridade por equipe
contagem_pri = df.groupby(["Grupo designado","Prioridade_Ordinal"]).size().reset_index(name="contagem_prioridade")
df_moda_p_times = (
    contagem_pri.sort_values("contagem_prioridade", ascending=False)
    .drop_duplicates(subset=["Grupo designado"], keep="first")
    [["Grupo designado","Prioridade_Ordinal"]]
    .rename(columns={"Prioridade_Ordinal": "Moda_Prioridade"})
)

# Resumo por equipe
def resumo_time(g):
    n = len(g)
    return pd.Series({
        "Total_Incidentes": n,
        "Pct_P1": round((g["Prioridade_Ordinal"]==1).sum()/n*100, 2),
        "Pct_P2": round((g["Prioridade_Ordinal"]==2).sum()/n*100, 2),
        "Pct_P3": round((g["Prioridade_Ordinal"]==3).sum()/n*100, 2),
        "Pct_P4": round((g["Prioridade_Ordinal"]==4).sum()/n*100, 2),
        "Pct_P5": round((g["Prioridade_Ordinal"]==5).sum()/n*100, 2),
        "Pct_P1_Total": round((g["Prioridade_Ordinal"]==1).sum()/total_registros*100, 2),
        "Pct_P2_Total": round((g["Prioridade_Ordinal"]==2).sum()/total_registros*100, 2),
        "Pct_P3_Total": round((g["Prioridade_Ordinal"]==3).sum()/total_registros*100, 2),
        "Pct_P4_Total": round((g["Prioridade_Ordinal"]==4).sum()/total_registros*100, 2),
        "Pct_P5_Total": round((g["Prioridade_Ordinal"]==5).sum()/total_registros*100, 2),
        "Pct_Manual":   round((g["Aberto por"]=="Manual").sum()/total_registros*100, 2),
        "Produtos_Dis": g["Produto"].nunique(),
        "Categorias_Dis": g["Categoria"].nunique(),
        "Itens_Configuracao_Dis": g["Item de configuração"].nunique(),
    })

df_resumo_times = df.groupby("Grupo designado").apply(resumo_time).reset_index()

df_times = df_resumo_times.merge(df_moda_p_times, on="Grupo designado", how="left")
df_times["Inc_por_Produto"]          = (df_times["Total_Incidentes"] / df_times["Produtos_Dis"]).round(2)
df_times["Inc_por_Categoria"]        = (df_times["Total_Incidentes"] / df_times["Categorias_Dis"]).round(2)
df_times["Inc_por_Item_Configuracao"]= (df_times["Total_Incidentes"] / df_times["Itens_Configuracao_Dis"]).round(2)

display(df_times.sort_values("Total_Incidentes", ascending=False).reset_index(drop=True))

In [ ]:
# Heatmap de especialização por prioridade dentro de cada time
pdf = df_times.set_index("Grupo designado")
pct_cols = ["Pct_P1","Pct_P2","Pct_P3","Pct_P4","Pct_P5"]

plt.figure(figsize=(12, 8))
sns.heatmap(pdf[pct_cols], annot=True, cmap="YlGnBu", fmt=".1f")
plt.title("Especialização por Prioridade (% dentro do time)")
plt.tight_layout()
plt.show()

### Incidentes por dia

In [ ]:
df["Data"] = df["Aberto"].dt.normalize()  # mantém como datetime (meia-noite) para merge posterior

df_diario = df.groupby("Data").apply(lambda g: pd.Series({
    "Total_Incidentes": len(g),
    "P1": (g["Prioridade"] == "1 - Crítica").sum(),
    "P2": (g["Prioridade"] == "2 - Alta").sum(),
    "P3": (g["Prioridade"] == "3 - Média").sum(),
    "P4": (g["Prioridade"] == "4 - Baixa").sum(),
    "P5": (g["Prioridade"] == "5 - Muito Baixa").sum(),
    "Possui_Produto":           g["Produto"].notna().sum(),
    "Possui_Item_Configuracao": g["Item de configuração"].notna().sum(),
    "Possui_Resolvido":         g["Resolvido"].notna().sum(),
    "Encerrados no Mesmo Dia":  (g["Encerrado"].dt.normalize() == g["Aberto"].dt.normalize()).sum(),
    "Duração_Média":            g["Duração"].mean(),
    "Possui_Solução":           g["Solução"].notna().sum(),
    "Solução_Contorno":         (g["Solução"] == "Contorno").sum(),
    "Solução_Definitiva":       (g["Solução"] == "Definitiva").sum(),
    "Aberto_Manual":            (g["Aberto por"] == "Manual").sum(),
    "Aberto_Monitoramento":     (g["Aberto por"] == "Monitoramento").sum(),
    "Possui_Incidente_Pai":     g["Incidente Pai"].notna().sum(),
    "Encerrado":                (g["Status"] == "Encerrado").sum(),
    "Encerrado_Automaticamente":(g["Status"] == "Encerrado Automaticamente").sum(),
    "Sem Intervenção":          (g["Status"] == "Sem Intervenção").sum(),
    "Entrou_KPI":               (g["Entrou para KPI?"] == "SIM").sum(),
    "KPI_Violado":              (g["KPI Violado?"] == "SIM").sum(),
    "Tempo_Pos_Resolução_Médio": g["Tempo_Pos_Resolução"].mean() if "Tempo_Pos_Resolução" in g.columns else np.nan,
})).reset_index().sort_values("Data")

print(f"Dias com incidentes: {len(df_diario)}")

In [ ]:
# Verificando se o total de dias com incidentes é igual ao intervalo total
print(len(df_diario) == intervalo_total)
print(f"Diferença: {len(df_diario) - intervalo_total} dias sem incidentes")

In [ ]:
# Preenchendo datas sem incidentes (left join com sequência completa de datas)
todas_datas = pd.DataFrame({
    "Data": pd.date_range(data_minima.normalize(), data_maxima.normalize(), freq="D")
})

df_diario = todas_datas.merge(df_diario, on="Data", how="left")

# Preencher colunas numéricas com 0 onde não havia incidentes
cols_num = df_diario.columns.difference(["Data"])
df_diario[cols_num] = df_diario[cols_num].fillna(0)

df_diario["tem_incidente"] = (df_diario["Total_Incidentes"] > 0).astype(int)

print(f"Diferença após completar datas: {len(df_diario) - intervalo_total - 1}")

In [ ]:
# Gráfico de Total de Incidentes por Dia
fig = px.line(
    df_diario,
    x="Data",
    y="Total_Incidentes",
    title="Total de Incidentes por Dia",
    markers=True
)
fig.update_layout(
    xaxis_title="Data",
    yaxis_title="Quantidade de Incidentes",
    hovermode="x unified"
)
fig.show()